# To Bugfix

prepare.py        | steps.py                  | pipeline/                | status
--------------------------------------------------------------------------
prepare_dataset() | download()                | build_manifest()         | X
                  |                           | download_methylation()   | X
                  |                           | initialize_audit_table() | X
                  |                           | build_metadata()         | X
                  |                           | build_biospecimen()      | X
                  |                           | update_metadata()        | X
                  |                           | update_biospecimen()     | X
                  | clean_data()              |                          | X
                  | load_raw_project()        |                          | X
                  | quality_control()         | load_annotation()        | X
                  |                           | sample_qc()              | X
                  |                           | probe_qc()               | X
                  |                           | clean_metadata()         | X
                  | preprocess()              | normalize()              | X
                  |                           | filter_variance()        | X
                  |                           | convert_to_mval()        | X
                  |                           | impute()                 | X
                  | save_project()            |                          | X
prepare_cohort()  | load_processed_project()  |                          |
                  | aggregate_cohort()        | cohort_aggregation()     |
                  | cohort_batch_correction() | batch_correct()          |
                  | aggregate_genes()         | gene_aggregation()       |
                  | winsorize()               |                          |
                  | split()                   |                          |

%load_ext autoreload
%autoreload 2


## Initial Set-up

In [ ]:
from methyltrain.config.loader import load_config
from methyltrain.fs.layout import ProjectLayout, CohortLayout

layout = ProjectLayout(
    project_name = "TCGA-ESCA",
    root_dir = "../data",
    raw_dir = "../data/raw/TCGA-ESCA",
    audit_table = "../data/metadata/TCGA-ESCA/TCGA-ESCA_audit_table.csv",
    metadata = "../data/metadata/TCGA-ESCA/TCGA-ESCA_metadata.csv",
    manifest = "../data/metadata/TCGA-ESCA/TCGA-ESCA_manifest.csv",
    status_log = "../data/metadata/TCGA-ESCA/TCGA-ESCA_status_log.csv",
    project_adata = "../data/processed/TCGA-ESCA_adata.h5ad"
)
layout.initialize()
layout.validate()

config = load_config(
    "/Volumes/FBI_Drive/MethylTrain/config/local_config.yaml"
)

## Current Debug

## Setup

In [1]:
from typing import Dict, Tuple

import anndata as ad
import pandas as pd

from methyltrain.config.loader import load_config
from methyltrain.fs.layout import CohortLayout

from methyltrain.api.steps import (
    download, 
    clean_data, 
    quality_control, 
    preprocess, 
    aggregate_cohort, 
    cohort_batch_correction,
    aggregate_genes,
    winsorize,
    split,
    load_raw_project,
    load_processed_project
)

projects = [
    "../data/processed/TCGA-CESC_adata.h5ad",
    "../data/processed/TCGA-CHOL_adata.h5ad",
    "../data/processed/TCGA-COAD_adata.h5ad",
    "../data/processed/TCGA-ESCA_adata.h5ad",
    "../data/processed/TCGA-GBM_adata.h5ad",
    "../data/processed/TCGA-KIRP_adata.h5ad",
    "../data/processed/TCGA-LUAD_adata.h5ad",
    "../data/processed/TCGA-OV_adata.h5ad",
    "../data/processed/TCGA-PAAD_adata.h5ad",
    "../data/processed/TCGA-UVM_adata.h5ad",
]

layout = CohortLayout(
    cohort_name = "pancancer",
    root_dir = "../data",
    project_list = projects,
    cohort_adata = "../data/processed/pancancer_cohort_adata.h5ad",
    train_adata = "../data/processed/pancancer_train_adata.h5ad",
    val_adata = "../data/processed/pancancer_val_adata.h5ad",
    test_adata = "../data/processed/pancancer_test_adata.h5ad"
)

In [14]:
config = load_config(
    "../config/pancancer_config.yaml"
)

In [2]:
# Ensure directories exist
layout.initialize()

In [3]:
# Load each processed project AnnData object 
project_adatas = [load_processed_project(path) 
                  for path in layout.project_list]

## Current Debug

### aggregate_cohort() -> cohort_aggregation()

In [4]:
from methyltrain.api.steps import aggregate_cohort

cohort_adata = aggregate_cohort(project_adatas, layout)

### cohort_batch_correction -> batch_correction()

In [6]:
import anndata as ad
import numpy as np

from inmoose.pycombat import pycombat_norm
from typing import Dict

In [8]:
adata = cohort_adata

In [9]:
X = np.array(adata.X)

# Ensure no NaNs remain in the AnnData object
if np.isnan(X).any():
    raise ValueError("Batch correction requires no Nans. Please impute "
                     "or remove missing values first.")

# Ensure data type is M-values, not beta values
if adata.uns['conversion'] != "m_value":
    raise ValueError("Batch correction should be performed on M-values. "
                     "Please convert from beta values to M-values first.")

In [15]:
# Ensure batch and covariate columns exist
batch_cfg = config.get('batch_correction', {})
batch_key = batch_cfg.get('batch_key', 'aliquot_id')
covariates = batch_cfg.get('covariates', ['project_id'])

for col in [batch_key] + covariates:
    if col not in adata.obs.columns: 
        raise ValueError(f"Column {col} not found in adata.obs. Required "
                         "for batch correction.")

In [16]:
# Prepare the batch vector and covariate matrix
batch = adata.obs[batch_key].astype(str).values
mod = adata.obs[covariates]

In [22]:
# Apply ComBat via the InMoose package
adata.X = pycombat_norm(counts = X.T, batch = batch, covar_mod = mod).T

ValueError: Batches 00129f7c-8240-4100-9a22-033abb70aba6, 0041a05f-411f-4d61-9a33-45a8fd2d0030, 0089e229-e06f-4d47-b1b9-b9e12d45ed04, 00ad1cb0-3b5c-4eaa-ae77-b66918b0b5dd, 00de8879-949c-457c-9780-0d96c1821dcc, 00edb826-8b89-41f2-a5c7-b5ee3b7fdb4e, 01130c26-9524-4242-ad73-e00430a090af, 0140291c-1e4e-4d13-90e2-b67dab13c5ef, 014b1799-6e35-464c-b418-5d9c99373bb3, 01740dac-84fc-489b-a1ff-de8944ef8910, 017adc07-bff6-4371-854b-48b4e1b2f9f7, 01a29e96-2485-4c8f-af59-12818592fb3e, 01a5bc8f-3e83-4456-b662-ee58beadc548, 01c2d915-949d-4aea-b9d8-b869b69ef9f6, 01d1b615-e45c-4191-b788-ebe717510a4e, 021e885c-37b2-440c-873c-de12257b226a, 023e9eeb-250d-4163-8406-436d8f844278, 025c6e3e-a7f3-4349-9cb8-a4165dc18c7b, 02976ce5-d7af-435b-8e27-5d0e6b353cff, 02ba9e8b-1918-497f-9af2-85503e2da18a, 02d78ea8-8fa5-4e63-aab7-497cb87805eb, 02ec6466-4a0d-4946-8f15-d8678507bb80, 02f102ca-7505-4a4a-974e-79f0f0bacf33, 03140683-6775-405f-b7b5-2f29bc47e713, 031f521e-8b65-4d54-8a3b-3be421471b54, 032f97ec-21d7-4fd1-9bda-a105a1